# Theme analysis Plots

We will do our plots with Flourish. We just need to create the csv and export them in their online plateform to obtain our plots.

In [1]:
import os
from pathlib import Path

import pandas as pd

from IPython.display import HTML

In [2]:
# General variables: paths
ROOT = Path("../../data/newyorker_caption_contest")
THEME_PATH = ROOT / "THEMES_RESULTS_V4" 

# Files paths
DOCS_PATH   = THEME_PATH / "docs_with_themes.csv"
TF_IDF_PATH = THEME_PATH / "TF_IDF_themes_scores.csv"
THEMES_PATH = THEME_PATH / "themes.csv"

print(f"THEME_PATH:  {THEME_PATH.resolve()}\n")
print(f"DOCS_PATH:   {DOCS_PATH.resolve()}")
print(f"TF_IDF_PATH: {TF_IDF_PATH.resolve()}")
print(f"THEMES_PATH: {THEMES_PATH.resolve()}")

THEME_PATH:  /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4

DOCS_PATH:   /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4/docs_with_themes.csv
TF_IDF_PATH: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4/TF_IDF_themes_scores.csv
THEMES_PATH: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V4/themes.csv


In [3]:
def load_csv(csv_path: Path, expected_cols) -> pd.DataFrame:
    """Load a csv file into Pandas DataFrame and check for expected columns."""
    
    df = pd.read_csv(csv_path)
    
    # check columns
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing expected columns: {missing}")
    
    return df

In [4]:
# load all content with themes extraction csv
doc_cols = ["doc_id", "contest_id", "kind", "text", "clean_text", "rank", "mean", "precision",
            "votes", "not_funny", "somewhat_funny", "funny", "theme_id", "theme_label"]
df_doc = load_csv(DOCS_PATH, expected_cols=doc_cols)
df_doc.head()

,doc_id,contest_id,kind,text,clean_text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,theme_id,theme_label
0,image_510,510,image,A man is relaxing on a city street. Others are...,man relax city street go business step pass ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,Train
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,congressman obstruction job,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,64,Election
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",mean middle class step,1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,53,Market
3,caption_510_2,510,caption,Does this suit make me look flat?,suit look flat,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,76,Clothing
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",right woman come know,3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,45,Marriage


In [5]:
# # load TF-IDF theme terms scores csv
# tfidf_cols = ["theme_id", "term", "sim", "local_freq", "global_freq", "distinctiveness", "score"]
# df_tfidf = load_csv(TF_IDF_PATH, expected_cols=tfidf_cols)
# df_tfidf.head()

In [ ]:
# # load the 80 themes id, terms and stats
# themes_cols = ["theme_id", "theme_label", "nb_instances", "mu_rank", "mu_mean", "mu_precision",
#                "mu_not_funny", "mu_somewhat_funny", "mu_funny", "TF_IDF_terms"]
# df_themes = load_csv(THEMES_PATH, expected_cols=themes_cols)
# df_themes.head()

## Theme relations between images and captions

Sketch:

<img src="../../figures/sketches/graph_sketch.jpg" width="400">

In [6]:
# --- Build Flourish heatmap links with labels + explicit ids ---
# Output columns: Source, Target, value, source_id, target_id

# Keep only what you need
cols = ["contest_id", "kind", "theme_id", "theme_label"]
df = df_doc[cols].copy()

# Light memory cleanup
df["contest_id"]  = pd.to_numeric(df["contest_id"], errors="coerce").astype("Int32")
df["theme_id"]    = pd.to_numeric(df["theme_id"], errors="coerce").astype("Int16")
df["theme_label"] = df["theme_label"].astype("string")

# Theme id -> label mapping (for Source/Target strings)
theme_map = (
    df[["theme_id", "theme_label"]]
    .dropna(subset=["theme_id", "theme_label"])
    .drop_duplicates(subset=["theme_id"])
    .set_index("theme_id")["theme_label"]
    .to_dict()
)

# All theme IDs (aim to include all 80)
all_ids = sorted([int(x) for x in theme_map.keys() if pd.notna(x)])
# If your IDs are 1..80 but some are missing from data, force the full set:
if len(all_ids) != 80 and all_ids and min(all_ids) >= 1 and max(all_ids) <= 80:
    all_ids = list(range(1, 81))

# Split: one image per contest
df_img = (
    df[df["kind"].eq("image")]
    .dropna(subset=["contest_id", "theme_id"])
    .drop_duplicates(subset=["contest_id"])
    .rename(columns={"theme_id": "img_theme_id", "theme_label": "img_theme_label"})
    [["contest_id", "img_theme_id"]]
)

df_cap = (
    df[df["kind"].eq("caption")]
    .dropna(subset=["contest_id", "theme_id"])
    .rename(columns={"theme_id": "cap_theme_id", "theme_label": "cap_theme_label"})
    [["contest_id", "cap_theme_id"]]
)

# Pair captions with their contest's image theme
pairs = df_cap.merge(df_img, on="contest_id", how="inner", validate="many_to_one")

# Aggregate counts by ids
links = (
    pairs.groupby(["img_theme_id", "cap_theme_id"], sort=False)
         .size()
         .reset_index(name="value")
         .rename(columns={"img_theme_id": "source_id", "cap_theme_id": "target_id"})
)

# Build full 80x80 grid so Flourish can show all themes (missing pairs -> 0)
full_grid = pd.MultiIndex.from_product([all_ids, all_ids], names=["source_id", "target_id"]).to_frame(index=False)

df_flourish = (
    full_grid.merge(links, on=["source_id", "target_id"], how="left")
             .fillna({"value": 0})
)
df_flourish["value"] = df_flourish["value"].astype(int)

# Build Source/Target labels like "T25: Manhattan" while keeping ids as separate cols
def _label(tid: int) -> str:
    lab = theme_map.get(int(tid), "")
    return f"T{int(tid)}: {lab}" if lab else f"T{int(tid)}"

df_flourish["Source"] = df_flourish["source_id"].apply(_label)
df_flourish["Target"] = df_flourish["target_id"].apply(_label)

# Final column order for Flourish
df_flourish = df_flourish[["Source", "Target", "value", "source_id", "target_id"]]

# Export if you want:
# df_flourish.to_csv("flourish_heatmap.csv", index=False)

df_flourish.head()


,Source,Target,value,source_id,target_id
0,T0: Dog,T0: Dog,5109,0,0
1,T0: Dog,T1: Fall Asleep,122,0,1
2,T0: Dog,T2: Year,352,0,2
3,T0: Dog,T3: Work Home,509,0,3
4,T0: Dog,T4: Clown,278,0,4


In [8]:
print(Path("../../data/newyorker_caption_contest/FLOURISH/flourish_alluvial.csv").resolve())

/Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/FLOURISH/flourish_alluvial.csv


In [9]:
df_flourish.to_csv("../../data/newyorker_caption_contest/FLOURISH/flourish_heatmap.csv", index=False)

In [ ]:
# Size reduction for readability

TOP_THEMES_PER_SIDE = 15   # keep top N nodes per side; rest -> removed
TOP_LINKS_PER_SOURCE = 5   # keep top K outgoing links per Source
MIN_VALUE = 1500             # drop links below this after aggregation
DROP_SELF_LINKS = False    # Source!=Target only if True

df_f = df_flourish.copy()

# optional: remove self-links
if DROP_SELF_LINKS:
    df_f = df_f[df_f["Source"] != df_f["Target"]].copy()

# keep only top nodes on each side (by total flow)
top_sources = (df_f.groupby("Source")["value"].sum().nlargest(TOP_THEMES_PER_SIDE).index)
top_targets = (df_f.groupby("Target")["value"].sum().nlargest(TOP_THEMES_PER_SIDE).index)

df_f = df_f[df_f["Source"].isin(top_sources) & df_f["Target"].isin(top_targets)].copy()

# drop tiny links
df_f = df_f[df_f["value"] >= MIN_VALUE].copy()

# keep only top K links per Source (within the retained nodes)
df_f = (df_f.sort_values(["Source", "value"], ascending=[True, False])
           .groupby("Source", as_index=False)
           .head(TOP_LINKS_PER_SOURCE)
           .reset_index(drop=True))

df_flourish_reduced = df_f

df_flourish_reduced.head()



In [ ]:
print("Length df_flourish:", len(df_flourish))
print("Length df_flourish_reduced:", len(df_flourish_reduced))

In [ ]:
df_flourish_reduced.to_csv("../../data/newyorker_caption_contest/FLOURISH/flourish_alluvial_reduced3.csv", index=False)

Results:

In [ ]:
# Display the Flourish Sankey diagram (with fixed width)
HTML("""
<div class="flourish-embed flourish-sankey" data-src="visualisation/26895805">
  <script src="https://public.flourish.studio/resources/embed.js"></script>
  <noscript>
    <img src="https://public.flourish.studio/visualisation/26895805/thumbnail" width="100%" alt="sankey visualization" />
  </noscript>
</div>
""")